# Strict Isomorphism

General idea: For $A_1=(Q_1,Σ,τ_1,(q_s)_1,F_1)$ and $A_2=(Q_2,Σ,τ_2,(q_s)_2,F_2)$

To be strictly isomorphic

--Need a bijection from $Q_1$ to $Q_2$, so must have same cardinalities ($|Q_1|=|Q_2|$), return False if not

--Need $f(F_1)=F_2$, since $f$ is bijection, must have $|F_1|=|F_2|$, return False if not

Now we can construct the mapping $f$

Note that we assumed that $Q={0,1,...,n-1}$ for some natural number $n$, if $f$ is a bijection, it is a permutation (as $Q_1=Q_2$)

--Want $f$ to map $(q_s)_1$ to $(q_s)_2$, start to build the mapping, set $f((q_s)_1)=(q_s)_2$

--Use Breadth-first search to construct the rest of the mapping

----Start with just (qs1,qs2) in a queue

----While queue

------Take the first thing as q1,q2, check if q1 in F1 == q2 in F2 (after construction, this will satisfy f(F1)

--Lastly check that the conditions $f(F_1)=F_2$ and $τ_2(f(q),a)=f(τ_1(q,a))$ for all $q∈Q_1$ and all $a∈Σ$ indeed holds


In [1]:
from collections import deque
def is_strict_isomorphism(A1,A2):
    """
    Tests if two DFAs A1=(Q1,Σ,τ1,qs1,F1) and A2=(Q2,Σ,τ2,qs2,F2) are stricly isomorphic.
    Assumes they have the common alphabet Σ.
    Assumes every state in Q is reachable from qs.

    Parameters:
    - A1, A2: Two DFAs
    Arguments of A1, A2:
    - Q1, Q2: some finite set of states
    - Σ: some finite alphabet (a set)
    - τ1, τ2: transition function τi:Qi×Σ→Qi for i=1,2 (a dict)
    - qs1,qs2: start state
    - F1, F2: set of accepting or final states

    Returns: 
    Boolean True or False
    """
    # Store the arguments of the DFAs in variables(if not already in this form)
    Q1,Σ,τ1,qs1,F1=A1
    Q2,Σ,τ2,qs2,F2=A2
    # Test the cardinalities of the Qs and the Fs are equal
    if len(Q1)!=len(Q2):
        return False
    if len(F1)!=len(F2):
        return False
    # Build the mapping, since assmued the Qs are {0,1,...,n-1}, the mapping will turn out to be a permutation.
    f=[-1]*len(Q1)   # Note f initially has -1 in every entry, not 0, as there's a vertex called 0
    # Map qs1 to qs2 i.e. f(qs1)=qs2
    f[qs1]=qs2
    # Now construct the rest of f using Breadth-first search, first make a queue containg the tuple (qs1,qs2)        
    queue=deque()
    queue.append((qs1,qs2))
    # While there are still things in the queue
    while queue:
        # Get the first thing of the queue, note q1 is in Q1 and q2 is in Q2
        q1,q2=queue.popleft()
        # Run through every number in the alphabet
        for a in Σ:
            # Use τ to get the next states, called ns
            ns1=τ1[(q1,a)]
            ns2=τ2[(q2,a)]
            # If ns1 is unmapped, map it to ns2 and throw in to the queue
            if f[ns1]==-1:
                f[ns1]=ns2
                queue.append((ns1,ns2))
    # Now check if f(F1)=F2, by checking every q in Q1 that is in F1 has f(q) in F2, and vice versa
    for q in Q1:
        if (q in F1)!=(f[q] in F2):
            return False
        # Check if τ2(f(q),a)=f(τ1(q,a)) for all q∈Q1 and all a∈Σ
        for a in Σ:
            if τ2[f[q],a]!=f[τ1[q,a]]:
                return False
    # If reaches here, then passed all the reqirements, so isomorphic
    return True
            



Below are functions to genereate DFAs and isomorpic/non-isomorphic DFAs by made by deepseek

Used to test 'is_strict_isomorphism'

In [2]:
import random

def generate_dfa(n, m, f_size):
    """
    Generate a random DFA A = (Q, sigma, tau, qs, F) with:
        Q = {0, 1, ..., n-1}
        sigma = {0, 1, ..., m-1}
        tau: dict with keys (state, symbol) -> next state
        qs = 0 (start state)
        F = set of f_size accepting states

    Ensures every state is reachable from the start state.
    """
    if f_size < 0 or f_size > n:
        raise ValueError("f_size must be between 0 and n")

    # Track available symbols per state (to avoid collisions when building the spanning tree)
    available = [set(range(m)) for _ in range(n)]

    # Transition function: dict (state, symbol) -> next_state
    tau = {}

    # Build a spanning tree from state 0 to all other states
    reachable = [0]          # states already in the tree
    remaining = set(range(1, n))   # states not yet added

    while remaining:
        # Pick a random state from reachable that still has free symbols
        candidates = [s for s in reachable if available[s]]
        u = random.choice(candidates)

        # Pick a random unused symbol for u
        a = random.choice(list(available[u]))

        # Pick a random target from the remaining states
        v = random.choice(list(remaining))

        # Assign the transition
        tau[(u, a)] = v
        available[u].remove(a)

        # Update sets
        reachable.append(v)
        remaining.remove(v)

    # Fill all remaining transitions arbitrarily
    for u in range(n):
        for a in range(m):
            if (u, a) not in tau:
                tau[(u, a)] = random.randint(0, n-1)

    # Choose accepting states
    F = set(random.sample(range(n), f_size))

    # Q and sigma as sets (as described)
    Q = set(range(n))
    sigma = set(range(m))
    qs = 0

    return Q, sigma, tau, qs, F

In [9]:
def isomorphic_dfa(dfa, perm=None):
    """
    Return a new DFA that is isomorphic to the input DFA by applying a state permutation.

    Parameters
    ----------
    dfa : tuple (Q, sigma, tau, qs, F)
        The original DFA.
    perm : dict, optional
        A dictionary mapping each old state to a new state.
        If None, a random permutation of the state set is generated.

    Returns
    -------
    new_dfa : tuple
        The isomorphic DFA with states relabeled according to the permutation.
    """
    Q, sigma, tau, qs, F = dfa
    n = len(Q)

    # Generate a random permutation if none is provided
    if perm is None:
        # Create a random permutation of the states (list where new_state = perm_list[old_state])
        perm_list = list(range(n))
        random.shuffle(perm_list)
        old_to_new = {old: new for old, new in enumerate(perm_list)}
    else:
        old_to_new = perm

    # Apply the permutation to all components
    new_tau = {}
    for (s, a), t in tau.items():
        new_tau[(old_to_new[s], a)] = old_to_new[t]

    new_qs = old_to_new[qs]
    new_F = {old_to_new[f] for f in F}
    new_Q = set(old_to_new.values())          # should equal the original set of integers

    return new_Q, sigma, new_tau, new_qs, new_F

def non_isomorphic_dfa(dfa, state=None, symbol=None, new_target=None):
    """
    Return a new DFA obtained by changing one transition of the input DFA.

    Parameters
    ----------
    dfa : tuple (Q, sigma, tau, qs, F)
        The original DFA.
    state : int, optional
        The source state of the transition to modify. If None, a random state is chosen.
    symbol : int, optional
        The alphabet symbol. If None, a random symbol is chosen.
    new_target : int, optional
        The new target state. If None, a random state from Q is chosen.

    Returns
    -------
    new_dfa : tuple
        The DFA with the specified transition replaced.
    """
    Q, sigma, tau, qs, F = dfa
    n = len(Q)

    # Choose random values if not specified
    if state is None:
        state = random.choice(list(Q))
    if symbol is None:
        symbol = random.choice(list(sigma))
    if new_target is None:
        new_target = random.choice(list(Q))

    # Create a modified copy of the transition dictionary
    new_tau = tau.copy()
    new_tau[(state, symbol)] = new_target

    return Q, sigma, new_tau, qs, F

In [19]:
for i in range(50):
    A1=generate_dfa(10,4,6)
    A2=isomorphic_dfa(A1)
    A3=non_isomorphic_dfa(A1)
    print(is_strict_isomorphism(A1,A2),is_strict_isomorphism(A1,A3))

True False
True False
True False
True False
True False
True False
True False
True False
True False
True False
True True
True True
True True
True False
True False
True True
True False
True False
True False
True True
True False
True True
True False
True False
True False
True False
True False
True False
True True
True False
True True
True False
True False
True False
True False
True False
True False
True False
True False
True False
True False
True False
True False
True False
True False
True False
True False
True False
True False
True False
